# InferenceOverviewSimExtended

Extended overfitting / cross-fitting simulation matching the original
Stata DML simulation (Ahrens `ddml_simulations sim_Overfit`, dgp 5 + dgp 3)
as closely as PyTorch allows.

Notebook mirror of `Python code/InferenceOverviewSimExtended.py` for
interactive use. Both files coexist; running this notebook does not
modify the .py.

**Smoke-test guard:** the bottom-of-notebook CSV write is set up so it
does NOT overwrite the canonical `Data/InferenceOverviewSimExtended.csv`
unless `n_rep >= 50`. Smoke runs land in `notes/` (gitignored).

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.linalg import cholesky, toeplitz
from sklearn.linear_model import LassoCV
import torch
import torch.nn as nn
import torch.optim as optim
from joblib import Parallel, delayed
import warnings
warnings.filterwarnings("ignore")

# Paths: when run from `Python code/` the canonical Data/ dir is one up.
HERE    = Path(os.getcwd()).resolve()
REPO    = HERE.parent if HERE.name == "Python code" else HERE
DATADIR = (REPO / "Data").resolve()
NOTEDIR = (REPO / "notes").resolve()
print(f"REPO    = {REPO}")
print(f"DATADIR = {DATADIR}")

In [ ]:
# Shared parameters
N        = 1000
P        = 50
ALPHA0   = 0.5
K_FOLDS  = 5

# X covariance: S_X[i,j] = 0.5^|i-j|; Cholesky used to draw X = Z @ SX_chol.
_SX_CHOL = cholesky(toeplitz(0.5 ** np.arange(P)))

# Linear DGP coefficient vector: beta_j = (0.9)^j for j = 1..p
_BETA = 0.9 ** np.arange(1, P + 1)


def _draw_X(rng):
    return rng.standard_normal((N, P)) @ _SX_CHOL


def _hetero_sigma(z):
    """Stata's `(1+z)^2` rescaled so its sample mean is 1, then sqrt."""
    s2 = (1.0 + z) ** 2
    return np.sqrt(s2 / s2.mean())

In [ ]:
# DGPs
def linear_dgp(rng):
    """DGP 5 (linear / approximately sparse)."""
    cy, cd = 0.189, 0.298
    X = _draw_X(rng)
    Xall = X @ _BETA
    v = rng.standard_normal(N)
    e = rng.standard_normal(N)

    sigd = _hetero_sigma(cd * Xall)
    D = cd * Xall + v * sigd

    sigy = _hetero_sigma(ALPHA0 * D + cy * Xall)
    Y = ALPHA0 * D + cy * Xall + e * sigy

    EDX = cd * Xall
    EYX = (ALPHA0 * cd + cy) * Xall
    return X, D, Y, EYX, EDX


def nonlinear_dgp(rng):
    """DGP 3 (nonlinear / indicator product)."""
    cy, cd = 1.42, 2.29
    X = _draw_X(rng)
    Xall = ((X[:, 0] > 0.3) & (X[:, 1] > 0.0) & (X[:, 2] > -1.0)).astype(np.float64)
    v = rng.standard_normal(N)
    e = rng.standard_normal(N)

    sigd = _hetero_sigma(cd * Xall)
    D = cd * Xall + v * sigd

    sigy = _hetero_sigma(ALPHA0 * D + cy * Xall)
    Y = ALPHA0 * D + cy * Xall + e * sigy

    EDX = cd * Xall
    EYX = (ALPHA0 * cd + cy) * Xall
    return X, D, Y, EYX, EDX

In [ ]:
# DNN: PyTorch port of sklearn MLPRegressor defaults
def fit_sklearn_like_mlp(X, y,
                          hidden=(20, 20),
                          alpha=1e-4,           # sklearn: L2 weight-decay coef
                          lr=1e-3,              # sklearn: learning_rate_init
                          max_iter=200,         # sklearn: max_iter
                          batch_size_cap=200,   # sklearn: batch_size = min(200, n_samples)
                          tol=1e-4,
                          n_iter_no_change=10):
    """Single-call MLP fit mirroring sklearn MLPRegressor behaviour with
    default settings. Returns the trained nn.Module."""
    Xa = np.asarray(X, dtype=np.float32)
    ya = np.asarray(y, dtype=np.float32).reshape(-1, 1)
    n, p = Xa.shape
    bs = min(batch_size_cap, n)

    layers = []
    prev = p
    for h in hidden:
        lin = nn.Linear(prev, h)
        nn.init.xavier_uniform_(lin.weight)   # sklearn uses Glorot-uniform
        nn.init.zeros_(lin.bias)
        layers += [lin, nn.ReLU()]
        prev = h
    out = nn.Linear(prev, 1)
    nn.init.xavier_uniform_(out.weight)
    nn.init.zeros_(out.bias)
    layers.append(out)
    model = nn.Sequential(*layers)

    opt = optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), eps=1e-8,
                      weight_decay=alpha)
    loss_fn = nn.MSELoss()

    Xt = torch.from_numpy(Xa)
    yt = torch.from_numpy(ya)

    best_loss = float("inf")
    no_imp = 0
    model.train()
    for _ in range(max_iter):
        perm = torch.randperm(n)
        running = 0.0
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            opt.zero_grad()
            l = loss_fn(model(Xt[idx]), yt[idx])
            l.backward()
            opt.step()
            running += float(l.item()) * idx.numel()
        epoch_loss = running / n

        # sklearn early-stop logic (training-loss based; > strict on count)
        if epoch_loss > best_loss - tol:
            no_imp += 1
        else:
            no_imp = 0
        if epoch_loss < best_loss:
            best_loss = epoch_loss
        if no_imp > n_iter_no_change:
            break

    return model


def predict_torch(model, X):
    model.eval()
    with torch.no_grad():
        Xt = torch.as_tensor(np.asarray(X, dtype=np.float32))
        return model(Xt).cpu().numpy().flatten()

In [ ]:
# Estimators
def oracle_alpha(Y, D, EYX, EDX):
    rY = Y - EYX
    rD = D - EDX
    return float(rD @ rY / (rD @ rD))


def _kfold_indices(n, K, rng):
    idx = np.arange(n)
    rng.shuffle(idx)
    return np.array_split(idx, K)


def dml_alpha_lasso(X, Y, D, rng):
    n = X.shape[0]
    folds = _kfold_indices(n, K_FOLDS, rng)
    rY = np.empty(n); rD = np.empty(n)
    for k in range(K_FOLDS):
        test  = folds[k]
        train = np.concatenate([folds[j] for j in range(K_FOLDS) if j != k])
        my = LassoCV(cv=5, alphas=50, max_iter=5000).fit(X[train], Y[train])
        md = LassoCV(cv=5, alphas=50, max_iter=5000).fit(X[train], D[train])
        rY[test] = Y[test] - my.predict(X[test])
        rD[test] = D[test] - md.predict(X[test])
    return float(rD @ rY / (rD @ rD))


def dml_alpha_dnn(X, Y, D, rng):
    n = X.shape[0]
    folds = _kfold_indices(n, K_FOLDS, rng)
    rY = np.empty(n); rD = np.empty(n)
    for k in range(K_FOLDS):
        test  = folds[k]
        train = np.concatenate([folds[j] for j in range(K_FOLDS) if j != k])
        my = fit_sklearn_like_mlp(X[train], Y[train])
        md = fit_sklearn_like_mlp(X[train], D[train])
        rY[test] = Y[test] - predict_torch(my, X[test])
        rD[test] = D[test] - predict_torch(md, X[test])
    return float(rD @ rY / (rD @ rD))


def full_dnn_alpha(X, Y, D):
    my = fit_sklearn_like_mlp(X, Y)
    md = fit_sklearn_like_mlp(X, D)
    rY = Y - predict_torch(my, X)
    rD = D - predict_torch(md, X)
    return float(rD @ rY / (rD @ rD))

In [ ]:
# Per-rep
def run_one_rep(ii):
    seed = 1234 + ii
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    torch.set_num_threads(1)

    Xa, Da, Ya, EYa, EDa = linear_dgp(rng)
    Xb, Db, Yb, EYb, EDb = nonlinear_dgp(rng)

    return {
        "lin_oracle":  oracle_alpha(Ya, Da, EYa, EDa),
        "lin_dml_las": dml_alpha_lasso(Xa, Ya, Da, rng),
        "lin_dml_dnn": dml_alpha_dnn(Xa, Ya, Da, rng),
        "nl_oracle":   oracle_alpha(Yb, Db, EYb, EDb),
        "nl_dml_las":  dml_alpha_lasso(Xb, Yb, Db, rng),
        "nl_dml_dnn":  dml_alpha_dnn(Xb, Yb, Db, rng),
        "nl_full_dnn": full_dnn_alpha(Xb, Yb, Db),
    }

In [ ]:
# Driver: set n_rep here, then run the cells below.
# n_rep = 3 is a quick smoke test; n_rep = 1000 is the canonical figure source.
n_rep  = 3
n_jobs = 1   # set to 8 (or os.cpu_count()) for multi-core

t0 = time.perf_counter()
if n_jobs == 1:
    results = []
    for ii in range(n_rep):
        results.append(run_one_rep(ii))
        if (ii + 1) <= 3 or (ii + 1) % max(1, n_rep // 10) == 0:
            elapsed = time.perf_counter() - t0
            per_rep = elapsed / (ii + 1)
            eta = per_rep * (n_rep - ii - 1)
            print(f"  rep {ii+1}/{n_rep}  elapsed={elapsed:.1f}s  per-rep={per_rep:.2f}s  ETA={eta:.0f}s",
                  flush=True)
else:
    results = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(run_one_rep)(ii) for ii in range(n_rep)
    )
elapsed = time.perf_counter() - t0
df = pd.DataFrame(results)
print(f"\nTotal wall: {elapsed:.1f}s ({elapsed/60:.2f} min) for {n_rep} reps")

In [ ]:
print(f"alpha_0 = {ALPHA0}")
print(f"  {'estimator':<14} {'mean':>8} {'sd':>8} {'bias':>9}")
for c in ("lin_oracle", "lin_dml_las", "lin_dml_dnn",
          "nl_oracle",  "nl_dml_las",  "nl_dml_dnn",  "nl_full_dnn"):
    v = df[c]
    print(f"  {c:<14} {v.mean():>8.3f} {v.std():>8.3f} {v.mean() - ALPHA0:>+9.3f}")
df.head()

In [ ]:
# Save raw results.
# IMPORTANT: small smoke runs (n_rep < 50) write to notes/ (gitignored)
# so the canonical 1000-rep CSV at Data/InferenceOverviewSimExtended.csv
# is NEVER overwritten by accident. Only set n_rep >= 50 when you mean
# to regenerate the canonical figure source.
if n_rep >= 50:
    out_dir = DATADIR
    out_name = "InferenceOverviewSimExtended.csv"
else:
    out_dir = NOTEDIR
    out_name = "InferenceOverviewSimExtended_smoke.csv"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / out_name
df.to_csv(csv_path, index=False)
print(f"Saved raw -> {csv_path}")
print("Run InferenceOverviewSimExtended_plots.ipynb to regenerate slide figures.")